# 🎬 AI 오디오리액티브 — AI가 음악을 듣고 영상을 그립니다

피아노 연주를 AI에게 들려주면, 음악의 비트·음량·음색 변화를 분석하여
**프레임 단위로 이미지를 생성**하고, 이를 이어붙여 영상을 만듭니다.

조용한 구간에서는 이미지가 부드럽게 흐르고, 클라이맥스에서는 격렬하게 변합니다.

**도구**: SD-Turbo (고속 1-step 이미지 생성) + TAESD (초소형 디코더) + librosa + ffmpeg

⏰ 이 노트북은 시간이 걸립니다 (10초 영상 기준 약 1분). 수업 후에 천천히 해보세요.

▶ 먼저 아래 셀을 실행해주세요. 설치에 2~3분 정도 걸립니다.

In [ ]:
%%capture
!apt-get -qq install -y ffmpeg > /dev/null 2>&1
!pip install -q diffusers transformers accelerate librosa soundfile matplotlib Pillow

import warnings
warnings.filterwarnings('ignore')

print('✅ 설치 완료!')

---
## 1. 피아노 연주 파일 올리기

▶ 영상을 만들 피아노 연주 파일을 업로드하세요.

STEP 1~2에서 만든 오디오를 사용하면, 본인의 연주가 영상이 되는 경험을 할 수 있습니다.
파일이 없으면 다음 셀에서 예시를 사용하세요.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
input_audio = list(uploaded.keys())[0]
print(f'✅ 파일이 업로드되었습니다: {input_audio}')

---
## 2. (선택) 예시 파일 사용

▶ 파일이 없으면 이 셀을 실행하세요. 쇼팽 녹턴 예시를 다운로드합니다.

In [ ]:
import urllib.request
import IPython.display as ipd

REPO_URL = 'https://raw.githubusercontent.com/joonhyungbae/pianokit/main/assets'
input_audio = 'piano_chopin.wav'

if not os.path.exists(input_audio):
    try:
        urllib.request.urlretrieve(f'{REPO_URL}/{input_audio}', input_audio)
        print(f'✅ 다운로드 완료: {input_audio}')
    except:
        print(f'⚠️ 다운로드 실패. 위 셀에서 직접 업로드해주세요.')

print(f'\n🎵 원본 오디오:')
if os.path.exists(input_audio):
    ipd.display(ipd.Audio(input_audio))

---
## 3. 음악 특성 분석

영상에 음악을 반영하려면, 먼저 음악에서 **어떤 변화가 일어나는지** 수치로 추출해야 합니다.
AI가 분석하는 세 가지 특성:

| 특성 | 의미 | 영상에서의 역할 |
|------|------|---------------|
| **RMS (음량)** | 소리의 크기 | 음량 클수록 이미지 변화 격렬 |
| **Spectral Centroid (음색)** | 소리의 밝기 | 고음 많을수록 밝은 분위기 |
| **Beat (비트)** | 박자 위치 | 비트마다 장면 전환 |

▶ 아래 셀을 실행하면 분석 결과를 그래프로 볼 수 있습니다.

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt

# 오디오 로드
y, sr = librosa.load(input_audio, sr=22050)
duration = len(y) / sr
print(f'🎵 오디오 길이: {duration:.1f}초')

# RMS (음량)
rms = librosa.feature.rms(y=y)[0]
rms_norm = (rms - rms.min()) / (rms.max() - rms.min() + 1e-8)

# Spectral Centroid (음색 밝기)
cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
cent_norm = (cent - cent.min()) / (cent.max() - cent.min() + 1e-8)

# Beat 감지
tempo, beats = librosa.beat.beat_track(y=y, sr=sr)
beat_times = librosa.frames_to_time(beats, sr=sr)

# 시각화
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

times = librosa.times_like(rms, sr=sr)

# 파형 + RMS
axes[0].plot(np.arange(len(y))/sr, y, alpha=0.3, linewidth=0.3, color='gray')
axes[0].plot(times, rms_norm, color='#FF6B6B', linewidth=2, label='RMS (음량)')
axes[0].set_ylabel('음량')
axes[0].legend()
axes[0].set_title('🔊 음량 변화 — 영상의 변화 강도를 결정')

# Spectral Centroid
times_c = librosa.times_like(cent, sr=sr)
axes[1].plot(times_c, cent_norm, color='#4ECDC4', linewidth=2, label='Spectral Centroid (음색)')
axes[1].set_ylabel('밝기')
axes[1].legend()
axes[1].set_title('✨ 음색 밝기 — 프롬프트 충실도를 결정')

# Beats
axes[2].vlines(beat_times, 0, 1, color='#FFE66D', linewidth=2, label=f'비트 ({len(beat_times)}개)')
axes[2].set_ylabel('비트')
axes[2].set_xlabel('시간 (초)')
axes[2].legend()
axes[2].set_title('🥁 비트 위치 — 장면 전환 시점')

plt.tight_layout()
plt.show()

print(f'\n📊 분석 결과: 템포 {tempo:.0f} BPM, {len(beat_times)}개 비트 감지')

---
## 4. 이미지 생성 AI 불러오기

**SD-Turbo**는 단 1스텝으로 이미지를 만드는 초경량 모델입니다.
**TAESD**는 초소형 디코더로, 이미지 변환 속도를 더욱 높여줍니다.

⏰ 모델 다운로드에 1~2분 걸립니다. ▶ 아래 셀을 실행하세요.

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image, AutoPipelineForImage2Image, AutoencoderTiny
from PIL import Image

print('🔄 SD-Turbo + TAESD 모델을 불러오는 중...')

# SD-Turbo: 860M params, ~3.4GB 다운로드
model_id = 'stabilityai/sd-turbo'
# 더 높은 품질을 원하면 아래 모델로 교체 (6.5GB 다운로드, 프레임당 ~800ms):
# model_id = 'stabilityai/sdxl-turbo'

pipe_txt2img = AutoPipelineForText2Image.from_pretrained(
    model_id, torch_dtype=torch.float16, variant='fp16'
)

# TAESD: 초소형 디코더로 교체 (디코딩 속도 대폭 향상)
pipe_txt2img.vae = AutoencoderTiny.from_pretrained(
    'madebyollin/taesd', torch_dtype=torch.float16
).to('cuda')

pipe_txt2img.to('cuda')

# img2img — 같은 컴포넌트 공유 (추가 VRAM 없음)
pipe_img2img = AutoPipelineForImage2Image.from_pipe(pipe_txt2img)

print('✅ SD-Turbo + TAESD 로딩 완료! (~3.5GB VRAM)')

---
## 5. 영상의 시각적 분위기 설정

▶ 영상이 어떤 느낌이면 좋겠는지 영어로 적어주세요.
이 프롬프트가 모든 프레임의 기본 분위기를 결정합니다.

**프롬프트 예시:**

| 프롬프트 | 분위기 |
|---------|-------|
| `abstract flowing shapes, dark blue and gold, digital art` | 추상적, 어두운 고급감 |
| `underwater world with bioluminescent creatures` | 신비로운 해저 세계 |
| `cosmic nebula with swirling colors` | 우주 성운 |

In [ ]:
prompt = "abstract flowing shapes, dark blue and gold, digital art"  # ← 여기를 수정하세요

print(f'🎨 프롬프트: "{prompt}"')

---
## 6. 어떻게 음악이 영상에 반영되나요?

3단계에서 분석한 음악 특성이 이미지 생성 파라미터에 매핑됩니다:

| 음악 특성 | → | 영상 파라미터 | 효과 |
|-----------|---|-------------|------|
| 음량 (RMS) | → | `strength` | 음량 클수록 이미지 변화 격렬 |
| 비트 (beat) | → | seed 변화 | 비트마다 새로운 이미지 전환 |

SDXL-Turbo는 `guidance_scale=0`으로 작동하므로 음색 매핑은 strength에 통합했습니다.

조용한 피아노 솔로 → 부드럽게 흐르는 영상

포르티시모 클라이맥스 → 화면이 격렬하게 변화

다음 셀에서 실제로 프레임을 생성합니다.

---
## 7. 프레임 생성

▶ 아래 셀을 실행하면 프레임을 하나씩 생성합니다.

⏰ SD-Turbo + TAESD는 프레임당 1스텝, 약 200ms — 10초 영상 (120프레임) 기준 **약 30초~1분** 소요

In [ ]:
from tqdm.auto import tqdm
from IPython.display import display, clear_output

duration_sec = 10  # ← 영상 길이 (초). 조절 가능
fps = 12           # ← 초당 프레임 수 (SD-Turbo가 빠르므로 12fps 가능)

total_frames = duration_sec * fps
print(f'🎬 총 {total_frames}개 프레임 생성 예정 ({duration_sec}초, {fps}fps)')

# 프레임 저장 디렉토리
os.makedirs('frames', exist_ok=True)

# 비트 시점을 프레임 번호로 변환
beat_frames = set([int(bt * fps) for bt in beat_times if bt < duration_sec])

# 첫 프레임: txt2img로 생성
seed = 42
generator = torch.Generator('cuda').manual_seed(seed)

print('🔄 첫 프레임 생성 중...')
init_image = pipe_txt2img(
    prompt=prompt,
    num_inference_steps=1,
    guidance_scale=0.0,
    generator=generator,
    width=512, height=512
).images[0]

init_image.save('frames/frame_0000.png')
prev_image = init_image

# 이후 프레임: img2img + 오디오 매핑
for frame_idx in tqdm(range(1, total_frames), desc='프레임 생성'):
    t = frame_idx / fps

    rms_idx = min(int(t / duration * len(rms_norm)), len(rms_norm) - 1)
    current_rms = rms_norm[rms_idx]

    strength = 0.3 + current_rms * 0.5

    if frame_idx in beat_frames:
        seed += 1

    generator = torch.Generator('cuda').manual_seed(seed)

    frame = pipe_img2img(
        prompt=prompt,
        image=prev_image,
        strength=min(strength, 0.85),
        guidance_scale=0.0,
        num_inference_steps=1,
        generator=generator,
    ).images[0]

    frame.save(f'frames/frame_{frame_idx:04d}.png')
    prev_image = frame

    if frame_idx % 20 == 0:
        print(f'  프레임 {frame_idx}/{total_frames} (t={t:.1f}s, rms={current_rms:.2f})')

print(f'\n✅ {total_frames}개 프레임 생성 완료!')

# 프레임 미리보기
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
preview_indices = [0, total_frames//3, total_frames*2//3, total_frames-1]
for ax, idx in zip(axes, preview_indices):
    img = Image.open(f'frames/frame_{idx:04d}.png')
    ax.imshow(img)
    ax.set_title(f'{idx/fps:.1f}초')
    ax.axis('off')
plt.suptitle('생성된 프레임 미리보기', fontsize=14)
plt.tight_layout()
plt.show()

---
## 8. 프레임 → 영상으로 합치기

생성된 프레임을 이어붙이고, 원본 음악을 입혀서 최종 영상을 만듭니다.

▶ 아래 셀을 실행하세요.

In [ ]:
import subprocess
from IPython.display import HTML
from base64 import b64encode

output_video = 'output_video.mp4'

# ffmpeg로 프레임 → 영상 + 오디오 합성
cmd = [
    'ffmpeg', '-y',
    '-framerate', str(fps),
    '-i', 'frames/frame_%04d.png',
    '-i', input_audio,
    '-t', str(duration_sec),
    '-c:v', 'libx264',
    '-pix_fmt', 'yuv420p',
    '-c:a', 'aac',
    '-shortest',
    output_video
]

print('🔄 영상을 합성하고 있습니다...')
result = subprocess.run(cmd, capture_output=True, text=True)

if os.path.exists(output_video):
    size_mb = os.path.getsize(output_video) / 1024 / 1024
    print(f'✅ 영상 합성 완료! ({size_mb:.1f}MB)')

    # 영상 재생
    with open(output_video, 'rb') as f:
        video_data = f.read()
    video_b64 = b64encode(video_data).decode()
    display(HTML(f'''
    <video width="512" controls>
        <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
    </video>
    '''))
else:
    print('⚠️ 영상 합성에 실패했습니다.')
    print(result.stderr[-500:] if result.stderr else '')

---
## 9. 결과 다운로드

▶ 완성된 영상을 다운로드하세요.

In [ ]:
from google.colab import files

if os.path.exists(output_video):
    files.download(output_video)
    print('✅ 다운로드 완료!')
else:
    print('⚠️ 영상 파일이 없습니다. 위 셀들을 먼저 실행해주세요.')